In [1]:
import pandas as pd
import numpy as np
import os
import math

# =========================================================
# CONFIG
# =========================================================
INPUT_DIR  = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_input_files"
OUTPUT_DIR = r"C:\Users\loci_\Desktop\trading_webapp\DATA"

RX_FILE = os.path.join(INPUT_DIR, "RX1_small.csv")
AD_FILE = os.path.join(INPUT_DIR, "AD1_small.csv")

START_NAV_USD = 10_000_000
ANNUAL_VOL = 0.20
TRADING_DAYS = 256

# fixed PDM as per your request
PDM_CONST = 1.6

# run both at 50/50
WEIGHTS = {"RX1": 0.5, "AD1": 0.5}

# your EWMA family
EWMA_SETUPS = [
    {"fast": 2,  "slow": 2*4,  "scaler": 10.6},
    {"fast": 4,  "slow": 4*4,  "scaler": 7.5},
    {"fast": 8,  "slow": 8*4,  "scaler": 5.3},
    {"fast": 16, "slow": 16*4, "scaler": 3.75},
    {"fast": 32, "slow": 32*4, "scaler": 2.65},
    {"fast": 64, "slow": 64*4, "scaler": 1.87},
]

# =========================================================
# HELPERS
# =========================================================
def calc_sharpe(series, annualisation=256):
    s = series.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) <= 2 or s.std() == 0:
        return np.nan
    return s.mean() / s.std() * np.sqrt(annualisation)

def compute_turnover(df, trading_days_per_year=256):
    if "trades" not in df.columns or "current_position" not in df.columns:
        return np.nan, np.nan, np.nan
    years = len(df) / trading_days_per_year
    avg_lots_traded_yearly = df["trades"].abs().sum() / years
    avg_abs_pos = df["current_position"].abs().mean()
    if avg_abs_pos == 0:
        return np.nan, avg_lots_traded_yearly, avg_abs_pos
    turnover = avg_lots_traded_yearly / (2 * avg_abs_pos)
    return turnover, avg_lots_traded_yearly, avg_abs_pos

# =========================================================
# CORE BACKTEST FOR ONE INSTRUMENT + ONE EWMA SETUP
# =========================================================
def run_one_ewma(df_raw, instr_name, fast, slow, scaler, weight, pdm):
    """
    df_raw: original instrument df with columns:
      Date, PX_CLOSE_1D, st_dev, Exchange rate, TICK_VALUE, TICK_SIZE (last 3 optional)
    """
    df = df_raw.copy()
    df = df.sort_values("Date").set_index("Date")

    price = df["PX_CLOSE_1D"].astype(float)
    stdev_price = df["st_dev"].astype(float)

    # FX — you said: IVV must be lower than ICV → so FX column is EUR per USD → divide
    if "Exchange rate" in df.columns:
        fx_series = df["Exchange rate"].astype(float)
    else:
        fx_series = pd.Series(1.0, index=df.index)

    # tick stuff
    if "TICK_VALUE" in df.columns and "TICK_SIZE" in df.columns:
        tick_value = float(df["TICK_VALUE"].iloc[0])
        tick_size  = float(df["TICK_SIZE"].iloc[0])
        MULT = tick_value / tick_size
    else:
        # fallback for RX/AD if missing
        tick_value = 1000.0
        tick_size  = 0.01
        MULT = tick_value / tick_size

    # returns for signal
    df["ret_pct_for_sig"] = price.pct_change()
    df["ret_net_for_var"] = price.diff()

    # variance = EWMA on squared net return
    decay = 2 / (36 + 1)  # your 36-day vol lookback
    variance = [np.nan] * len(df)
    ret_net_vals = df["ret_net_for_var"].values
    # find first non-nan
    start_idx = np.where(~np.isnan(ret_net_vals))[0][0]
    first_sq = ret_net_vals[start_idx] ** 2
    variance[start_idx] = first_sq
    prev_var = first_sq
    for i in range(start_idx + 1, len(df)):
        rn = ret_net_vals[i]
        rn2 = 0.0 if np.isnan(rn) else rn * rn
        v = decay * rn2 + (1 - decay) * prev_var
        variance[i] = v
        prev_var = v
    df["variance"] = variance
    df["stdev_variance"] = np.sqrt(df["variance"])

    # EWMA fast / slow (this is what's changing per run)
    df["ewma_fast"] = price.ewm(span=fast, adjust=False).mean()
    df["ewma_slow"] = price.ewm(span=slow, adjust=False).mean()
    df["raw_crossover"] = df["ewma_fast"] - df["ewma_slow"]

    # vol adj
    df["vol_adj_crossover"] = df["raw_crossover"] / df["stdev_variance"]

    # forecast for this speed
    df["scaled_forecast"] = df["vol_adj_crossover"] * scaler
    df["capped_forecast"] = df["scaled_forecast"].clip(-20, 20)

    # signal layer Sharpe
    df["forecast_x_return"] = df["capped_forecast"].shift(1) * df["ret_pct_for_sig"]

    # modular framework
    df["price_volatility"] = (stdev_price / price * 100).round(2)
    df["one_pct_move"] = price * 0.01
    df["block_value_eur"] = df["one_pct_move"] * MULT
    df["icv_eur"] = df["price_volatility"] * df["block_value_eur"]
    # ✅ your last correction: IVV must be lower than ICV → divide
    df["ivv"] = df["icv_eur"] / fx_series

    # iterative sizing
    nav_list = []
    dcv_list = []  # daily cash vol target
    target_contracts_list = []
    trades_list = []
    current_pos_list = []
    strat_ret_list = []
    vol_scaler_list = []
    subsystem_pos_list = []
    port_instr_pos_list = []

    prev_nav = START_NAV_USD
    prev_dcv = prev_nav * ANNUAL_VOL / np.sqrt(TRADING_DAYS)
    prev_target = 0

    for i, (date, row) in enumerate(df.iterrows()):
        px_today = row["PX_CLOSE_1D"]
        if i == 0:
            delta_price = 0.0
        else:
            px_yday = df["PX_CLOSE_1D"].iloc[i - 1]
            delta_price = px_today - px_yday

        # 1) today's risk budget = yesterday's NAV
        dcv_today = prev_dcv

        # 2) vol scaler
        ivv_today = row["ivv"]
        if np.isnan(ivv_today) or ivv_today == 0:
            vol_scaler_today = np.nan
        else:
            vol_scaler_today = dcv_today / ivv_today

        # 3) subsystem pos
        cap_f = row["capped_forecast"]
        if np.isnan(cap_f) or np.isnan(vol_scaler_today):
            subsystem_pos = 0.0
        else:
            subsystem_pos = (cap_f * vol_scaler_today) / 10.0

        # 4) apply instrument weight + PDM (constant, your choice)
        port_instr_pos = subsystem_pos * weight * pdm

        # 5) round to contracts
        target_contracts = int(round(port_instr_pos))
        trades = target_contracts - prev_target

        # 6) PnL (carry only, no real exec)
        carry_pnl_eur = prev_target * delta_price * MULT
        fx_today = fx_series.loc[date]
        carry_pnl_usd = carry_pnl_eur / fx_today  # because fx > 1 → EUR per USD

        pnl_usd = carry_pnl_usd
        nav_today = prev_nav + pnl_usd
        strat_ret = pnl_usd / prev_nav if prev_nav != 0 else 0.0

        # store
        nav_list.append(nav_today)
        dcv_list.append(dcv_today)
        target_contracts_list.append(target_contracts)
        trades_list.append(trades)
        current_pos_list.append(target_contracts)
        strat_ret_list.append(strat_ret)
        vol_scaler_list.append(vol_scaler_today)
        subsystem_pos_list.append(subsystem_pos)
        port_instr_pos_list.append(port_instr_pos)

        # roll
        prev_nav = nav_today
        prev_dcv = nav_today * ANNUAL_VOL / np.sqrt(TRADING_DAYS)
        prev_target = target_contracts

    # write to df
    df["nav_usd"] = nav_list
    df["daily_cash_vol_target"] = dcv_list
    df["volatility_scaler"] = vol_scaler_list
    df["subsystem_position"] = subsystem_pos_list
    df["portfolio_instr_position"] = port_instr_pos_list
    df["target_contracts"] = target_contracts_list
    df["current_position"] = current_pos_list
    df["trades"] = trades_list
    df["strategy_ret"] = strat_ret_list

    # sharpes
    raw_sharpe = calc_sharpe(df["forecast_x_return"])
    exec_sharpe = calc_sharpe(df["strategy_ret"])

    # turnover
    turnover, yr_traded, avg_pos = compute_turnover(df)

    # save per instrument/speed
    out_name = f"{instr_name}_ewma{fast}_{slow}.csv"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    df.reset_index().to_csv(out_path, index=False)

    return {
        "instrument": instr_name,
        "fast": fast,
        "slow": slow,
        "scaler": scaler,
        "raw_sharpe": raw_sharpe,
        "exec_sharpe": exec_sharpe,
        "turnover": turnover,
        "avg_yearly_lots": yr_traded,
        "avg_abs_pos": avg_pos,
        "csv": out_path,
    }

# =========================================================
# LOAD INSTRUMENT FILES (WITH CORRECT COLUMN NAMES)
# =========================================================
rx_df_raw = pd.read_csv(RX_FILE)
ad_df_raw = pd.read_csv(AD_FILE)

# make sure Date is datetime
rx_df_raw["Date"] = pd.to_datetime(rx_df_raw["Date"], dayfirst=True, errors="coerce")
ad_df_raw["Date"] = pd.to_datetime(ad_df_raw["Date"], dayfirst=True, errors="coerce")

rx_df_raw = rx_df_raw.dropna(subset=["Date"]).sort_values("Date")
ad_df_raw = ad_df_raw.dropna(subset=["Date"]).sort_values("Date")

# =========================================================
# RUN ALL SPEEDS FOR BOTH INSTRUMENTS
# =========================================================
results = []

for setup in EWMA_SETUPS:
    f = setup["fast"]
    s = setup["slow"]
    sc = setup["scaler"]

    print(f"\n=== Running EWMA {f}/{s} (scaler={sc}) ===")

    rx_res = run_one_ewma(
        rx_df_raw,
        "RX1",
        fast=f,
        slow=s,
        scaler=sc,
        weight=WEIGHTS["RX1"],
        pdm=PDM_CONST,
    )
    ad_res = run_one_ewma(
        ad_df_raw,
        "AD1",
        fast=f,
        slow=s,
        scaler=sc,
        weight=WEIGHTS["AD1"],
        pdm=PDM_CONST,
    )

    print(f"RX1 → raw Sharpe={rx_res['raw_sharpe']:.3f}, exec Sharpe={rx_res['exec_sharpe']:.3f}, turnover={rx_res['turnover']:.2f}")
    print(f"AD1 → raw Sharpe={ad_res['raw_sharpe']:.3f}, exec Sharpe={ad_res['exec_sharpe']:.3f}, turnover={ad_res['turnover']:.2f}")

    results.append(rx_res)
    results.append(ad_res)

# =========================================================
# PRINT SUMMARY
# =========================================================
print("\n================= SUMMARY (RX1 + AD1) =================")
print("instr | fast | slow | scaler | raw_sharpe | exec_sharpe | turnover")
for r in results:
    print(
        f"{r['instrument']:4s} | {r['fast']:4d} | {r['slow']:4d} | {r['scaler']:6.2f} | "
        f"{(r['raw_sharpe'] if not np.isnan(r['raw_sharpe']) else float('nan')):10.3f} | "
        f"{(r['exec_sharpe'] if not np.isnan(r['exec_sharpe']) else float('nan')):12.3f} | "
        f"{(r['turnover'] if not np.isnan(r['turnover']) else float('nan')):8.2f}"
    )

print("\n✅ Done. Check CSVs in:", OUTPUT_DIR)



=== Running EWMA 2/8 (scaler=10.6) ===
RX1 → raw Sharpe=0.053, exec Sharpe=0.010, turnover=58.52
AD1 → raw Sharpe=-0.791, exec Sharpe=-0.983, turnover=66.34

=== Running EWMA 4/16 (scaler=7.5) ===
RX1 → raw Sharpe=-0.567, exec Sharpe=-0.345, turnover=30.50
AD1 → raw Sharpe=-0.917, exec Sharpe=-1.097, turnover=37.36

=== Running EWMA 8/32 (scaler=5.3) ===
RX1 → raw Sharpe=-1.250, exec Sharpe=-0.996, turnover=16.86
AD1 → raw Sharpe=-1.036, exec Sharpe=-1.063, turnover=19.17

=== Running EWMA 16/64 (scaler=3.75) ===
RX1 → raw Sharpe=-1.345, exec Sharpe=-1.059, turnover=13.33
AD1 → raw Sharpe=-1.109, exec Sharpe=-0.874, turnover=12.00

=== Running EWMA 32/128 (scaler=2.65) ===
RX1 → raw Sharpe=-1.156, exec Sharpe=-0.965, turnover=11.74
AD1 → raw Sharpe=-1.064, exec Sharpe=-0.786, turnover=9.75

=== Running EWMA 64/256 (scaler=1.87) ===
RX1 → raw Sharpe=-0.999, exec Sharpe=-0.925, turnover=10.55
AD1 → raw Sharpe=-0.887, exec Sharpe=-0.714, turnover=8.42

================= SUMMARY (RX1 + AD